# Projeto 2 — Aprendizado de Máquina: Olist

**Disciplina:** T326 - Ciência dos Dados  
**Professor:** Caio Ponte  
**Turma:** 16/17  
**Equipe:** (mesma do Projeto 1)

---
## Etapa 1 — Definição do Problema (peso 10%)

### 1.1 Contexto da Olist

A **Olist** é uma empresa brasileira de tecnologia para e-commerce, com sede em Curitiba/PR. Ela atua como um **ecossistema de soluções**: conecta pequenos e médios vendedores aos principais marketplaces do Brasil (Mercado Livre, Americanas, Magalu, Amazon, Shopee etc.), oferecendo sistema de gestão (ERP), integração de canais, **logística/envio de pedidos** e soluções financeiras.

Com **+60 mil clientes** e operação em escala nacional, um dos pontos críticos do negócio é a **logística de entrega**: a Olist promete ao vendedor e ao comprador prazos confiáveis de entrega. Atrasos geram:

- insatisfação do cliente e queda no **NPS/nível de avaliação** (reviews);
- custos maiores de suporte e reenvio;
- perda de competitividade dos lojistas na plataforma.

Por isso, **prever quanto tempo um pedido leva para chegar ao cliente** é um problema de alto valor de negócio: permite antecipar gargalos, dimensionar transportadoras, comunicar prazos mais realistas e priorizar pedidos em risco.

### 1.2 O problema de regressão

| | |
|---|---|
| **Tipo de problema** | Regressão |
| **Cada amostra** | Um **pedido** (`order_id`) da Olist |
| **Variável alvo** | Tempo de entrega até o cliente, em **dias**: `order_delivered_customer_date - order_purchase_timestamp` |
| **Variáveis preditoras** | Características do pedido no momento da compra: itens, preço, frete, categoria do produto, peso/dimensões, estado de origem/destino, meio de pagamento, etc. |

**Por que regressão?** O alvo é uma variável contínua (número de dias), não uma classe.

**Observação importante:** só pedidos com status `delivered` e com data de entrega preenchida possuem alvo calculável. Pedidos cancelados/em andamento serão filtrados na etapa de pré-processamento.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

DATA = "dados/"

customers  = pd.read_csv(DATA + "olist_customers_dataset.csv")
geolocation = pd.read_csv(DATA + "olist_geolocation_dataset.csv")
items      = pd.read_csv(DATA + "olist_order_items_dataset.csv")
payments   = pd.read_csv(DATA + "olist_order_payments_dataset.csv")
reviews    = pd.read_csv(DATA + "olist_order_reviews_dataset.csv")
orders     = pd.read_csv(DATA + "olist_orders_dataset.csv")
products   = pd.read_csv(DATA + "olist_products_dataset.csv")
sellers    = pd.read_csv(DATA + "olist_sellers_dataset.csv")
categories = pd.read_csv(DATA + "product_category_name_translation.csv")

tabelas = {
    "orders": orders, "customers": customers, "items": items,
    "payments": payments, "reviews": reviews, "products": products,
    "sellers": sellers, "geolocation": geolocation, "categories": categories,
}

for nome, df in tabelas.items():
    print(f"{nome:12s} {df.shape[0]:>9,} linhas x {df.shape[1]} colunas")

In [ ]:
orders.head()

In [ ]:
print("Status dos pedidos:")
print(orders["order_status"].value_counts())

**Prévia do alvo** (apenas pedidos `delivered`, que têm data de entrega registrada):

In [ ]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

entregues = orders[orders["order_status"] == "delivered"].copy()
entregues["dias_ate_a_entrega"] = (
    entregues["order_delivered_customer_date"] - entregues["order_purchase_timestamp"]
).dt.total_seconds() / 86400

print(f"Pedidos entregues (com alvo possível): {len(entregues):,} de {len(orders):,}")
entregues["dias_ate_a_entrega"].describe()

### 1.3 Estrutura do dataset e relacionamento entre as tabelas

O dataset público da Olist é **relacional** (9 tabelas CSV). A tabela central é `orders`:

```
                         customers
                              │ customer_id
                              ▼
  payments ──order_id──► orders ◄──order_id── reviews
                          │ order_id
                          ▼
                        items ──product_id──► products ──category──► categories (pt→en)
                          │ seller_id
                          ▼
                        sellers

  customers/sellers ──zip_code_prefix──► geolocation (lat/lng)
```

| Tabela | Chave principal | Papel |
|---|---|---|
| `orders` | `order_id` | **Núcleo**: datas de compra, envio, entrega e previsão |
| `customers` | `customer_id` | Localização do cliente (estado/cidade de destino) |
| `items` | `order_id` + `product_id` + `seller_id` | Itens do pedido: preço, frete, vendedor |
| `products` | `product_id` | Categoria, peso e dimensões do produto |
| `sellers` | `seller_id` | Localização do vendedor (estado de origem) |
| `payments` | `order_id` | Forma de pagamento e parcelas |
| `reviews` | `order_id` | Nota de satisfação (possível feature pós-entrega / análise) |
| `geolocation` | `zip_code_prefix` | Coordenadas para mapas (EDA) |
| `categories` | `product_category_name` | Tradução da categoria PT → EN |

In [ ]:
chaves = {
    "orders": ["order_id", "customer_id"],
    "customers": ["customer_id", "customer_unique_id"],
    "items": ["order_id", "product_id", "seller_id"],
    "payments": ["order_id"],
    "reviews": ["order_id", "review_id"],
    "products": ["product_id", "product_category_name"],
    "sellers": ["seller_id"],
    "geolocation": ["geolocation_zip_code_prefix"],
    "categories": ["product_category_name"],
}

for nome, cols in chaves.items():
    unicos = ", ".join(f"{c}={tabelas[nome][c].nunique():,}" for c in cols)
    print(f"{nome:12s} | {unicos}")

**Tabelas escolhidas para o modelo (justificativa):**

- **`orders`** — obrigatória: define a amostra (1 pedido) e contém as datas necessárias para calcular o alvo.
- **`items`** — traz características do pedido no momento da compra: quantidade de itens, preço, frete e o vendedor responsável.
- **`customers`** — origem do cálculo do destino: estado/cidade do cliente é um dos fatores que mais influenciam o prazo.
- **`sellers`** — estado do vendedor permite derivar a **distância origem→destino** (feature forte para logística).
- **`products`** — peso e dimensões afetam o tipo de transporte e o frete/prazo.
- **`payments`** — forma de pagamento e parcelamento podem correlacionar-se com perfil/pedido (usada como feature adicional).

**`geolocation`** é usada principalmente na **EDA** (mapas), **`reviews`** é pós-entrega (não deve vazar informação do alvo no momento da previsão) e **`categories`** é um dicionário de tradução auxiliar.

### 1.4 Desafio real e impacto

- **Sazonalidade:** Black Friday, Natal e datas comemorativas concentram pedidos e estouram prazos.
- **Rede logística heterogênea:** combinação de transportadoras, regiões com infraestrutura diferente (ex.: interior x capitais, Norte x Sul) e grandes distâncias.
- **Variação entre vendedores:** tempo de despacho do lojista e tipo de produto (frágil, pesado, grande) mudam o prazo real.
- **Prazo prometido x real:** o `order_estimated_delivery_date` é uma estimativa da própria Olist — o desafio é **superar ou explicar o erro dessa estimativa**.

**Impacto prático:** um modelo de previsão de tempo de entrega permite: (1) prometer prazos mais realistas na compra, (2) acionar suporte antes do atraso, (3) redistribuir carga entre transportadoras e (4) reduzir churn de vendedores e reclamações nos reviews.

### Resumo da Etapa 1

- **Contexto:** Olist = ecossistema de e-commerce/logística brasileiro; prever prazo de entrega é crítico para experiência e custo.
- **Problema:** regressão — amostra = 1 pedido; alvo = dias entre compra e entrega ao cliente.
- **Tabelas principais:** `orders` (núcleo) + `items`, `customers`, `sellers`, `products`, `payments`; `geolocation`/`reviews` com papéis específicos.
- **Desafio:** sazonalidade, heterogeneidade da rede logística e distância origem→destino; impacto em NPS, custo e retenção de vendedores.

---
## Etapa 2 — Pré-processamento dos Dados (peso 30%)

Itens obrigatórios contemplados:
1. **Engenharia de features** — montagem da tabela de modelagem a partir dos joins
2. **Filtragem e limpeza** — pedidos sem alvo, valores inválidos e outliers extremos
3. **Tratamento de valores faltantes** — `SimpleImputer` (mediana / moda)
4. **Normalização/Padronização** — `RobustScaler` (robusto a outliers)
5. **Variáveis categóricas** — `OneHotEncoder`

Tudo aplicado via `ColumnTransformer` (pipeline do sklearn), pronto para a etapa de treino.


### 2.1 Engenharia de features

Cada amostra = 1 pedido (`entregue`). Alvo: **`dias_ate_a_entrega`**.

| Origem | Features |
|---|---|
| `orders` | prazo estimado, mês/dia/hora da compra |
| `items` | nº de itens, preço total, frete total, nº de produtos e vendedores |
| `products` | peso e dimensões médios, categoria do produto |
| `payments` | valor pago, máx. de parcelas, nº de pagamentos, tipo |
| `sellers` + `customers` + `geolocation` | distância média vendedor→cliente (km), estados de origem e destino |


In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

# --- alvo: pedidos entregues ---
df = orders[orders["order_status"] == "delivered"].copy()
df["dias_ate_a_entrega"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# --- itens do pedido ---
it = items.groupby("order_id").agg(
    n_itens=("order_item_id", "count"),
    preco_total=("price", "sum"),
    frete_total=("freight_value", "sum"),
    n_produtos=("product_id", "nunique"),
    n_vendedores=("seller_id", "nunique"),
).reset_index()

# --- características do produto (média dos itens do pedido) ---
ip = items[["order_id", "product_id"]].merge(products, on="product_id")
prod = ip.groupby("order_id").agg(
    peso_medio=("product_weight_g", "mean"),
    comp_medio=("product_length_cm", "mean"),
    alt_medio=("product_height_cm", "mean"),
    larg_medio=("product_width_cm", "mean"),
    categoria=("product_category_name", "first"),
).reset_index()

# --- pagamento ---
pay = payments.groupby("order_id").agg(
    valor_pagamento=("payment_value", "sum"),
    max_parcelas=("payment_installments", "max"),
    n_pagamentos=("payment_sequential", "count"),
    tipo_pagamento=("payment_type", "first"),
).reset_index()

# --- distância vendedor → cliente (Haversine via geolocation) ---
oc = orders[["order_id", "customer_id"]].merge(
    customers[["customer_id", "customer_state", "customer_zip_code_prefix"]],
    on="customer_id",
)
cliente = oc[["order_id", "customer_state"]].drop_duplicates("order_id")

pair = items[["order_id", "seller_id"]].merge(
    sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]], on="seller_id"
).merge(oc, on="order_id")

geo = geolocation.groupby("geolocation_zip_code_prefix").agg(
    lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean")
)
pair = pair.merge(geo, left_on="seller_zip_code_prefix", right_index=True)
pair = pair.merge(geo, left_on="customer_zip_code_prefix", right_index=True,
                  suffixes=("_v", "_c"))

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

pair["dist_km"] = haversine(pair["lat_v"], pair["lng_v"], pair["lat_c"], pair["lng_c"])
dist = pair.groupby("order_id").agg(
    dist_media_km=("dist_km", "mean"),
    estado_vendedor=("seller_state", "first"),
).reset_index()

# --- features temporais conhecidas na compra ---
df["prazo_estimado_dias"] = (
    df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400
df["mes_compra"] = df["order_purchase_timestamp"].dt.month
df["dia_semana"] = df["order_purchase_timestamp"].dt.dayofweek
df["hora_compra"] = df["order_purchase_timestamp"].dt.hour

# --- tabela final de modelagem ---
df = (df.merge(it, on="order_id")
        .merge(prod, on="order_id")
        .merge(pay, on="order_id")
        .merge(dist, on="order_id")
        .merge(cliente, on="order_id"))

print(f"Tabela de modelagem: {df.shape[0]:,} pedidos x {df.shape[1]} colunas")

In [ ]:
df.head()

### 2.2 Filtragem e limpeza dos dados

- remover pedidos sem alvo (datas ausentes)
- remover valores inválidos (`dias_ate_a_entrega` ≤ 0)
- remover outliers extremos do alvo (acima do **percentil 99**) — valores de 60–200 dias são ruído/logística atípica e distorcem a regressão


In [ ]:
n0 = len(df)
relatorio = {}

df = df.dropna(subset=["dias_ate_a_entrega"])
relatorio["sem_alvo (NaN)"] = n0 - len(df)

antes = len(df)
df = df[df["dias_ate_a_entrega"] > 0]
relatorio["alvo <= 0"] = antes - len(df)

q99 = df["dias_ate_a_entrega"].quantile(0.99)
antes = len(df)
df = df[df["dias_ate_a_entrega"] <= q99]
relatorio[f"outliers > p99 ({q99:.1f} dias)"] = antes - len(df)

print("Registros removidos:")
for k, v in relatorio.items():
    print(f"  {k:35s} {v:>6,}")
print(f"\nBase final: {len(df):,} pedidos")
df["dias_ate_a_entrega"].describe()

### 2.3 Tratamento de valores faltantes

Conferência dos NaNs restantes — a imputação em si entra no pipeline (`SimpleImputer`):
- numéricos → **mediana**
- categóricos → **moda**


In [ ]:
nan_antes = df.isna().sum()
nan_antes[nan_antes > 0]

### 2.4 Normalização e codificação (pipeline)

- **Numéricas:** `SimpleImputer(mediana)` + `RobustScaler` (não sofre com os outliers remanescentes)
- **Categóricas:** `SimpleImputer(moda)` + `OneHotEncoder`


In [ ]:
num_features = [
    "n_itens", "preco_total", "frete_total", "n_produtos", "n_vendedores",
    "peso_medio", "comp_medio", "alt_medio", "larg_medio",
    "valor_pagamento", "max_parcelas", "n_pagamentos",
    "dist_media_km", "prazo_estimado_dias",
    "mes_compra", "dia_semana", "hora_compra",
]
cat_features = ["categoria", "tipo_pagamento", "estado_vendedor", "customer_state"]
alvo = "dias_ate_a_entrega"

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]), num_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_features),
])

X = df[num_features + cat_features]
y = df[alvo]

X_processed = preprocessor.fit_transform(X)
print(f"Matriz processada: {X_processed.shape[0]:,} amostras x {X_processed.shape[1]} features (após OneHot)")

### 2.5 Resumo das features selecionadas


In [ ]:
resumo = pd.DataFrame({
    "feature": num_features + cat_features,
    "tipo": (["numérica"] * len(num_features)) + (["categórica"] * len(cat_features)),
    "origem": (
        ["items"] * 5 + ["products"] * 4 + ["payments"] * 3 +
        ["geolocation+sellers+customers"] + ["orders"] * 4 +
        ["products", "payments", "sellers", "customers"]
    ),
    "tratamento": (
        ["imputação mediana + RobustScaler"] * len(num_features) +
        ["imputação moda + OneHotEncoder"] * len(cat_features)
    ),
})
resumo["feature"].name = None
print(f"Total: {len(resumo)} features | alvo: {alvo}\n")
resumo.to_csv("dados/resumo_features.csv", index=False)
resumo

In [ ]:
# base limpa salva para as próximas etapas (EDA e treino)
df.to_csv("dados/df_modelo.csv", index=False)
print(f"Salvo: dados/df_modelo.csv ({len(df):,} linhas)")
print(f"Colunas: {list(df.columns)}")